## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">A Deep Research agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!
            </span>
        </td>
    </tr>
</table>

In [15]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool, set_tracing_disabled
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
import os
import httpx
from bs4 import BeautifulSoup
from sendgrid.helpers.mail import Mail, Email, To, Content
from IPython.display import display, Markdown
from typing import Dict, Literal
import smtplib
from email.mime.text import MIMEText

In [10]:
load_dotenv(override=True)

if os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENAI_API_KEY"] = os.getenv("OPENROUTER_API_KEY")
    os.environ["OPENAI_BASE_URL"] = os.getenv("OPENROUTER_API_BASE", "https://openrouter.ai/api/v1")
    # Traces go to api.openai.com; OpenRouter keys are not valid there (401).
    set_tracing_disabled(True)

## OpenAI Hosted Tools

OpenAI Agents SDK includes the following hosted tools:

The `WebSearchTool` lets an agent search the web.  
The `FileSearchTool` allows retrieving information from your OpenAI Vector Stores.  
The `ComputerTool` allows automating computer use tasks like taking screenshots and clicking.

**OpenRouter:** Hosted `WebSearchTool` is not supported (the API only accepts `function` / `custom` tools). When `OPENROUTER_API_KEY` is set, this notebook uses a `web_search` **function tool** that scrapes DuckDuckGo HTML instead.

### Important note - API charge of WebSearchTool

This is costing me 2.5 cents per call for OpenAI WebSearchTool. That can add up to $2-$3 for the next 2 labs. We'll use free and low cost Search tools with other platforms, so feel free to skip running this if the cost is a concern. Also student Christian W. pointed out that OpenAI can sometimes charge for multiple searches for a single call, so it could sometimes cost more than 2.5 cents per call.

Costs are here: https://platform.openai.com/docs/pricing#web-search

In [11]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

_use_openrouter_web_fallback = bool(os.getenv("OPENROUTER_API_KEY"))


@function_tool
async def web_search(query: str) -> str:
    """Search the web and return titles and snippets from top DuckDuckGo results."""
    async with httpx.AsyncClient(
        timeout=30.0,
        follow_redirects=True,
        headers={
            "User-Agent": "Mozilla/5.0 (compatible; AgentsCourse/1.0)",
            "Content-Type": "application/x-www-form-urlencoded",
        },
    ) as client:
        response = await client.post(
            "https://html.duckduckgo.com/html/",
            data={"q": query},
        )
        response.raise_for_status()
    soup = BeautifulSoup(response.text, "lxml")
    lines: list[str] = []
    for res in soup.select("div.result")[:10]:
        a = res.select_one("a.result__a")
        if not a:
            continue
        title = a.get_text(strip=True)
        snip = res.select_one(".result__snippet")
        snippet = snip.get_text(strip=True) if snip else ""
        lines.append(f"- {title}: {snippet}".strip())
    if not lines:
        return f"(No results parsed for {query!r}. Try a shorter query or check your network.)"
    return "\n".join(lines)


_search_tools = (
    [web_search]
    if _use_openrouter_web_fallback
    else [WebSearchTool(search_context_size="low")]
)

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=_search_tools,
    model="openai/gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

In [12]:
message = "Latest AI Agent frameworks in 2025"

with trace("Search"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))

In 2025, several AI agent frameworks have emerged as leaders in the field, each offering unique features and capabilities tailored for specific applications. Notable frameworks include LangChain, AutoGen, and CrewAI, which are highlighted for their user-friendly designs and versatility in various environments. Comparative analyses across multiple sources indicate that these frameworks excel in developer support, documentation, and adaptability to different use cases, from small-scale projects to enterprise-level deployments.

Additionally, frameworks such as BabyAGI and MetaGPT are being recognized for enhancing autonomous performance and automation capabilities. The focus is on production-ready solutions that have undergone real-world testing, with performance benchmarks and cost analyses available from organizations deploying these systems. As the market evolves, comprehensive guides are emerging to help developers choose the right tools based on attributes like integration support and scalability.

Overall, the landscape of AI agent frameworks in 2025 is characterized by an emphasis on robust performance, ease of use, and a focus on real-world applicability, providing a range of options for developers looking to leverage artificial intelligence in diverse applications.

### As always, take a look at the trace

https://platform.openai.com/traces

### We will now use Structured Outputs, and include a description of the fields

In [13]:
# See note above about cost of WebSearchTool

HOW_MANY_SEARCHES = 3

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

# Use Pydantic to define the Schema of our response - this is known as "Structured Outputs"
# With massive thanks to student Wes C. for discovering and fixing a nasty bug with this!

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")

    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")


planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model="openai/gpt-4o-mini",
    output_type=WebSearchPlan,
)

In [14]:

message = "Latest AI Agent frameworks in 2025"

with trace("Search"):
    result = await Runner.run(planner_agent, message)
    print(result.final_output)

searches=[WebSearchItem(reason='To find the most recent AI agent frameworks released in 2025, this search will cover news articles, announcements, and summaries of developments in AI technologies.', query='latest AI agent frameworks 2025'), WebSearchItem(reason='This search targets specific conferences or publications in AI where advancements and frameworks may have been presented in 2025, providing insights from industry leaders.', query='2025 AI conferences AI agent frameworks'), WebSearchItem(reason='To explore comparisons and analyses of different AI agent frameworks available in 2025, this search will help identify key technological trends and dominant frameworks being utilized.', query='AI agent frameworks comparison 2025')]


In [16]:

def _email_provider() -> Literal["gmail", "sendgrid"]:
    """Prefer Gmail SMTP when configured; otherwise SendGrid. Set EMAIL_PROVIDER to force one."""
    forced = (os.getenv("EMAIL_PROVIDER") or "").strip().lower()
    if forced == "gmail":
        if not (os.getenv("GMAIL_APP_PASSWORD") and os.getenv("GMAIL_ADDRESS")):
            raise ValueError("EMAIL_PROVIDER=gmail requires GMAIL_ADDRESS and GMAIL_APP_PASSWORD in .env")
        return "gmail"
    if forced == "sendgrid":
        if not os.getenv("SENDGRID_API_KEY"):
            raise ValueError("EMAIL_PROVIDER=sendgrid requires SENDGRID_API_KEY in .env")
        if not os.getenv("SENDGRID_FROM_EMAIL") or not os.getenv("SENDGRID_TO_EMAIL"):
            raise ValueError("EMAIL_PROVIDER=sendgrid requires SENDGRID_FROM_EMAIL and SENDGRID_TO_EMAIL in .env")
        return "sendgrid"
    if os.getenv("GMAIL_APP_PASSWORD") and os.getenv("GMAIL_ADDRESS"):
        return "gmail"
    if os.getenv("SENDGRID_API_KEY"):
        if not os.getenv("SENDGRID_FROM_EMAIL") or not os.getenv("SENDGRID_TO_EMAIL"):
            raise ValueError(
                "SendGrid needs SENDGRID_FROM_EMAIL and SENDGRID_TO_EMAIL — or add GMAIL_ADDRESS + GMAIL_APP_PASSWORD to use Gmail SMTP instead."
            )
        return "sendgrid"
    raise ValueError(
        "Set up email in .env: GMAIL_ADDRESS + GMAIL_APP_PASSWORD (Gmail), "
        "or SENDGRID_API_KEY + SENDGRID_FROM_EMAIL + SENDGRID_TO_EMAIL (SendGrid)."
    )


def send_lab_email(subject: str, body: str, *, subtype: Literal["plain", "html"] = "plain") -> None:
    """Send one message via Gmail or SendGrid (see _email_provider)."""
    prov = _email_provider()
    if prov == "gmail":
        user = os.environ["GMAIL_ADDRESS"].strip()
        pwd = os.environ["GMAIL_APP_PASSWORD"].replace(" ", "")
        to_addr = (os.getenv("GMAIL_TO_EMAIL") or user).strip()
        msg = MIMEText(body, subtype, "utf-8")
        msg["Subject"] = subject
        msg["From"] = user
        msg["To"] = to_addr
        with smtplib.SMTP("smtp.gmail.com", 587) as smtp:
            smtp.starttls()
            smtp.login(user, pwd)
            smtp.sendmail(user, [to_addr], msg.as_string())
        return
    from_addr = os.environ.get("SENDGRID_FROM_EMAIL")
    to_addr = os.environ.get("SENDGRID_TO_EMAIL")
    if not from_addr or not to_addr:
        raise ValueError("SendGrid needs SENDGRID_FROM_EMAIL and SENDGRID_TO_EMAIL in .env")
    content_type = "text/html" if subtype == "html" else "text/plain"
    sg = sendgrid.SendGridAPIClient(api_key=os.environ["SENDGRID_API_KEY"])
    mail = Mail(Email(from_addr), To(to_addr), subject, Content(content_type, body)).get()
    response = sg.client.mail.send.post(request_body=mail)
    if response.status_code >= 400:
        raise RuntimeError(f"SendGrid HTTP {response.status_code}: {response.body}")


In [17]:
# @function_tool
# def send_email(subject: str, html_body: str) -> Dict[str, str]:
#     """ Send out an email with the given subject and HTML body """
#     sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
#     from_email = Email("ed@edwarddonner.com") # Change this to your verified email
#     to_email = To("ed.donner@gmail.com") # Change this to your email
#     content = Content("text/html", html_body)
#     mail = Mail(from_email, to_email, subject, content).get()
#     sg.client.mail.send.post(request_body=mail)
#     return "success"



@function_tool
def send_email(subject: str, html_body: str)->Dict[str,str]:
    """Send an email with the given subject and HTML body to the configured recipient."""
    send_lab_email(subject, html_body, subtype="html")
    return {"status":"success"}

In [18]:
send_email

FunctionTool(name='send_email', description='Send an email with the given subject and HTML body to the configured recipient.', params_json_schema={'properties': {'subject': {'title': 'Subject', 'type': 'string'}, 'html_body': {'title': 'Html Body', 'type': 'string'}}, 'required': ['subject', 'html_body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x11409e700>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

In [19]:
INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the 
report converted into clean, well presented HTML with an appropriate subject line."""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model="gpt-4o-mini",
)



In [20]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData,
)

### The next 3 functions will plan and execute the search, using planner_agent and search_agent

In [21]:
async def plan_searches(query: str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

### The next 2 functions write a report and email it

In [22]:
async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    return result.final_output

async def send_email(report: ReportData):
    """ Use the email agent to send an email with the report """
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return report

### Showtime!

In [23]:
query ="Latest AI Agent frameworks in 2025"

with trace("Research trace"):
    print("Starting research...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    await send_email(report)  
    print("Hooray!")




Starting research...
Planning searches...
Will perform 3 searches
Searching...
Finished searching
Thinking about report...
Finished writing report
Writing email...
Email sent
Hooray!


### As always, take a look at the trace

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">Congratulations on your progress, and a request</h2>
            <span style="color:#00cc00;">You've reached an important moment with the course; you've created a valuable Agent using one of the latest Agent frameworks. You've upskilled, and unlocked new commercial possibilities. Take a moment to celebrate your success!<br/><br/>Something I should ask you -- my editor would smack me if I didn't mention this. If you're able to rate the course on Udemy, I'd be seriously grateful: it's the most important way that Udemy decides whether to show the course to others and it makes a massive difference.<br/><br/>And another reminder to <a href="https://www.linkedin.com/in/eddonner/">connect with me on LinkedIn</a> if you wish! If you wanted to post about your progress on the course, please tag me and I'll weigh in to increase your exposure.
            </span>
        </td>
    </tr>